In [1]:
from dotenv import load_dotenv
from src.db.chroma_db import ChromaDb
from src.models.openai_provider import OpenAILLMProvider, OpenAIEmbeddingProvider
from src.prompts import timeframe_detection_system, timeframe_detection_user
from src.models.schemas import TimeframeDetection
from src.services.nkod_data_processor import NkodDataProcessor
from src.db.graph_db import GraphDb
from src.db.sq_lite import SqLite
from datetime import date
from src.services.language_detector import LanguageDetector
from src.services.nkod_query_matcher import NkodQueryMatcher
from src.services.timeframe_detector import TimeframeDetector


load_dotenv()

/home/lamossta/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## Downloading and creating SPARQL endpoint for the NKOD metadata

In [2]:
nkod_data_processor = NkodDataProcessor("nkod")
graph_db = GraphDb(nkod_data_processor.catalog_name)
sq_lite = SqLite(nkod_data_processor.metadata_sql_path)

#nkod_data_processor.download_catalog_metadata()
#nkod_data_processor.create_metadata_csv(graph_db)
#nkod_data_processor.create_metadata_sql(sq_lite)

## Indexing the keywords, titles and descriptions from the NKOD metadata (TODO: matching a dataset)

In [3]:
openai_embeddings = OpenAIEmbeddingProvider(model_name="text-embedding-3-small")
chroma_db = ChromaDb(nkod_data_processor.vectordb_path)
#nkod_data_processor.index_catalog_metadata(sq_lite, openai_embeddings, chroma_db, verbose=True)
print(chroma_db.list_collections())

['nkod_keywords_en', 'nkod_descriptions_en', 'nkod_keywords_cs', 'nkod_descriptions_cs', 'nkod_titles_en', 'nkod_titles_cs']


## Language detection, Timeframe detection and Query matching

In [7]:
input_query = "události říčany"

model_name ="gpt-5"
openai_llm = OpenAILLMProvider(
    model_name=model_name,
    temperature=1.0
)

timeframe_detection = TimeframeDetector().detect(input_query, openai_llm, model_name)
language_detection = LanguageDetector().detect(input_query, openai_llm, model_name)
print(f"Language: {language_detection}, Timeframe: {timeframe_detection}")

k = 50
nkod_query_matcher = NkodQueryMatcher(input_query)
print(nkod_query_matcher.high_k_intersection(k, chroma_db, nkod_data_processor, language_detection.language, openai_embeddings))
print(nkod_query_matcher.get_matching_titles(5, chroma_db, nkod_data_processor, language_detection.language.value, openai_embeddings))

Language: cs, Timeframe: Timeframe not specified
[]
['https://data.gov.cz/zdroj/datové-sady/00240702/1425538386', 'https://data.gov.cz/zdroj/datové-sady/00246875/1168825388', 'https://data.gov.cz/zdroj/datové-sady/00253472/1139317447', 'https://data.gov.cz/zdroj/datové-sady/44992785/2a20aa19a367b61a345d164fb421f9c4', 'https://data.gov.cz/zdroj/datové-sady/00281999/1137821589']
